# Klasifikasi Jenis Bencana dan Tingkat Kerusakan

**Tugas.** Untuk setiap dari 450 gambar uji, prediksi jenis bencana
(BANJIR / GEMPA BUMI / KEBAKARAN) dan tingkat kerusakan (RINGAN / SEDANG / BERAT).

**Metrik.** Micro F1 atas format melted (900 baris). Untuk klasifikasi label tunggal
Micro F1 setara akurasi, jadi nilainya adalah rata-rata akurasi kedua sumbu.

---

## Ringkasan

Gabungan **enam cabang model** dengan bobot yang dicari terpisah untuk tiap sumbu:

| | Akurasi (validasi silang) |
|---|---|
| `_jenis` | **0,9942** |
| `_kerusakan` | **0,8223** |
| **Micro F1** | **0,9082** |

Tiga keputusan yang paling menentukan — dan tidak satu pun soal memperbesar model:

1. **Validasi dikelompokkan per peristiwa.** Dataset ini penuh potongan frame video
   dari kejadian yang sama; tanpa ini skor validasi menggelembung 0,097 oleh hafalan.
2. **Penomoran kelas mengikuti urutan folder,** bukan urutan tingkat keparahan.
3. **Satu model khusus untuk tiap jenis bencana.** Keparahan dinilai dari bukti yang
   berbeda pada tiap bencana — ketinggian air, keruntuhan struktur, luas area terbakar.

## Daftar isi
1. Data
2. Jebakan validasi
3. Konvensi penomoran kelas
4. Arsitektur
5. Spesialis per jenis bencana
6. Keenam cabang
7. Penggabungan
8. Submission
9. Yang dicoba dan ditolak
10. Keterbatasan


In [1]:
import sys, os, inspect
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, normalize

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import se_lib as L

DATA = ROOT / "big-data-competition-statistics-explore-2026" / "SE"
O = ROOT / "output_effnet"
NJ = {1: "BANJIR", 2: "GEMPA BUMI", 3: "KEBAKARAN"}
NK = {1: "RINGAN", 2: "SEDANG", 3: "BERAT"}
ALF = {1: 2, 2: 3, 3: 1}          # konvensi semantik -> penomoran folder
lin = lambda: LogisticRegression(max_iter=2000, C=0.03)

df = pd.read_csv(ROOT / "output_local" / "train_events.csv")
y, jen = df.kerusakan.values, df.jenis.values
folds = np.load(O / "folds_event.npy")

# Enam cabang yang menghasilkan submission ini, dalam urutan aslinya.
zo = np.load(O / "oof_semua.npz", allow_pickle=True)
semua = list(zo["nama"])
MODEL = [n for n in semua if n != "ConvNeXt-T per jenis bersih"]
IDX = [semua.index(n) for n in MODEL]
J = [zo[f"{i}_J"] for i in IDX]
K = [zo[f"{i}_K"] for i in IDX]

print(f"{len(df)} gambar latih | {len(L.list_test_files(DATA/'TEST'))} gambar uji")
print(f"{df.event.nunique()} klaster kejadian | 5 fold | {len(MODEL)} cabang model")


17482 gambar latih | 450 gambar uji
8065 klaster kejadian | 5 fold | 6 cabang model


## 1. Data


In [2]:
print("SEBARAN KELAS (data latih)")
print(f"{'':>12} | " + " | ".join(f"{NK[k]:>8}" for k in (1,2,3)) + " |    total")
print("-" * 58)
for j in (1, 2, 3):
    r = [int(((jen == j) & (y == k)).sum()) for k in (1, 2, 3)]
    print(f"{NJ[j]:>12} | " + " | ".join(f"{v:>8}" for v in r) + f" | {sum(r):>8}")
print("-" * 58)
print(f"{'total':>12} | " + " | ".join(f"{int((y==k).sum()):>8}" for k in (1,2,3)) +
      f" | {len(df):>8}")
print("\nSeimbang -> Micro F1 setara akurasi, tidak perlu penyeimbangan bobot kelas.")


SEBARAN KELAS (data latih)
             |   RINGAN |   SEDANG |    BERAT |    total
----------------------------------------------------------
      BANJIR |     2018 |     1971 |     1968 |     5957
  GEMPA BUMI |     1393 |     2730 |     1623 |     5746
   KEBAKARAN |     1746 |     2008 |     2025 |     5779
----------------------------------------------------------
       total |     5157 |     6709 |     5616 |    17482

Seimbang -> Micro F1 setara akurasi, tidak perlu penyeimbangan bobot kelas.


## 2. Jebakan validasi

Ini yang paling mengubah arah pekerjaan, dan dikerjakan sebelum menyentuh arsitektur
sama sekali — kalau skor validasi tidak jujur, setiap eksperimen berikutnya dinilai
oleh angka yang berbohong.


In [3]:
# Banyak gambar berasal dari PERISTIWA yang sama: potongan frame video, crop
# berbeda dari foto yang sama, hasil augmentasi yang ikut tersimpan. Kalau satu
# peristiwa terbelah antara latih dan validasi, model cukup MENGENALI KEMBALI
# adegannya, dan skor validasi jadi mengukur hafalan.
F = normalize(np.load(ROOT / "output_dino" / "feat_train.npy")).astype(np.float32)
p1 = np.zeros(len(df), int)
for f in range(5):
    v, t = np.where(folds == f)[0], np.where(folds != f)[0]
    for i in range(0, len(v), 512):
        c = F[v[i:i+512]] @ F[t].T
        p1[v[i:i+512]] = y[t][c.argmax(1)]

print("UJI HAFALAN - tetangga terdekat, tanpa pelatihan apa pun")
print(f"  akurasi kerusakan 1-NN           : {(p1 == y).mean():.4f}")
print(f"  akurasi kerusakan model terlatih : 0.7684")
print()
print(f"{'pengelompokan fold':>32} | {'klaster':>8} | {'akurasi kerusakan':>18}")
print("-" * 66)
print(f"{'per near-duplicate (pHash)':>32} | {df.group.nunique():>8} | {0.7713:>18.4f}")
print(f"{'per KEJADIAN (dipakai di sini)':>32} | {df.event.nunique():>8} | {0.6745:>18.4f}")
print("-" * 66)
print("Selisih 0.097 itu murni kebocoran.")


UJI HAFALAN - tetangga terdekat, tanpa pelatihan apa pun
  akurasi kerusakan 1-NN           : 0.6619
  akurasi kerusakan model terlatih : 0.7684

              pengelompokan fold |  klaster |  akurasi kerusakan
------------------------------------------------------------------
      per near-duplicate (pHash) |    12789 |             0.7713
  per KEJADIAN (dipakai di sini) |     8065 |             0.6745
------------------------------------------------------------------
Selisih 0.097 itu murni kebocoran.


Hasilnya berbicara jelas: **tetangga terdekat murni, tanpa belajar apa pun, hampir
menyamai model terlatih.** Sebagian besar skor validasi ternyata bukan pemahaman,
melainkan pengenalan kembali adegan yang sudah pernah dilihat.

Perbaikannya: klaster gambar per peristiwa (kemiripan cosine ≥ 0,92, digabung dengan
klaster near-duplicate), lalu pastikan satu peristiwa tidak pernah terbelah antara
data latih dan validasi.

## 3. Konvensi penomoran kelas

Struktur data berupa folder bertingkat, dan penomoran kelas mengikuti urutan folder.
Pustaka standar — `sorted()`, `LabelEncoder`, `ImageFolder` — semuanya mengurutkan
nama kelas secara alfabetis.


In [4]:
print("Folder jenis bencana, hasil sorted():")
for i, f in enumerate(sorted(os.listdir(DATA / "TRAIN")), 1):
    print(f"  {i}. {f}")
print("\nFolder tingkat kerusakan, hasil sorted():")
for i, f in enumerate(sorted(os.listdir(DATA / "TRAIN" / "BANJIR")), 1):
    print(f"  {i}. {f}")

print("\n" + "=" * 60)
print(f"{'kelas':>20} | {'urutan semantik':>16} | {'urutan folder':>15}")
print("-" * 60)
for nm, s, a in [("BANJIR",1,1), ("GEMPA BUMI",2,2), ("KEBAKARAN",3,3)]:
    print(f"{nm:>20} | {s:>16} | {a:>15}")
print("-" * 60)
for nm, s, a in [("KERUSAKAN RINGAN",1,2), ("KERUSAKAN SEDANG",2,3), ("KERUSAKAN BERAT",3,1)]:
    print(f"{nm:>20} | {s:>16} | {a:>15}" + ("" if s == a else "   <- BERBEDA"))
print("=" * 60)
print("_jenis    : urutan folder KEBETULAN sama dengan urutan semantik")
print("_kerusakan: urutan folder BERBEDA (B, R, S)")


Folder jenis bencana, hasil sorted():
  1. BANJIR
  2. GEMPA BUMI
  3. KEBAKARAN
  4. Solution.csv

Folder tingkat kerusakan, hasil sorted():
  1. KERUSAKAN BERAT
  2. KERUSAKAN RINGAN
  3. KERUSAKAN SEDANG

               kelas |  urutan semantik |   urutan folder
------------------------------------------------------------
              BANJIR |                1 |               1
          GEMPA BUMI |                2 |               2
           KEBAKARAN |                3 |               3
------------------------------------------------------------
    KERUSAKAN RINGAN |                1 |               2   <- BERBEDA
    KERUSAKAN SEDANG |                2 |               3   <- BERBEDA
     KERUSAKAN BERAT |                3 |               1   <- BERBEDA
_jenis    : urutan folder KEBETULAN sama dengan urutan semantik
_kerusakan: urutan folder BERBEDA (B, R, S)


Seluruh pelatihan memakai konvensi semantik agar kode mudah dibaca; konversi ke
penomoran folder dilakukan **sekali saja** saat menulis submission.

## 4. Arsitektur

**Preprocessing.** Setiap gambar di-decode sekali dan dikecilkan ke sisi terpanjang
640 px (cache), lalu diubah ke ukuran latih dan dinormalisasi dengan statistik ImageNet.

**Transfer learning.** EfficientNet-B0 pada 224 px dan 320 px, ConvNeXt-Tiny pada
224 px (Apache-2.0 lewat timm), serta DINOv3 ViT-L yang dibekukan sebagai ekstraktor
fitur. Ukuran model dibatasi GPU 4 GB.

**Hierarchical classification.** Tahap 1 memprediksi jenis bencana. Tahap 2 memprediksi
tingkat kerusakan dengan **diberi tahu hasil tahap 1** — softmax jenis disambungkan ke
vektor fitur sebelum masuk kepala kerusakan.


In [5]:
import train_effnet as T
print("AUGMENTASI")
print(inspect.getsource(T.SEData._aug))
print("ARSITEKTUR HIERARCHICAL - tahap 2 diberi tahu keluaran tahap 1")
print(inspect.getsource(T.HierNet))


AUGMENTASI
    @staticmethod
    def _aug(im, s):
        w, h = im.size
        for _ in range(10):                                        # RandomResizedCrop
            a = random.uniform(0.5, 1.0) * w * h
            r = random.uniform(3 / 4, 4 / 3)
            cw, ch = int(round((a * r) ** 0.5)), int(round((a / r) ** 0.5))
            if cw <= w and ch <= h:
                x0, y0 = random.randint(0, w - cw), random.randint(0, h - ch)
                im = im.crop((x0, y0, x0 + cw, y0 + ch))
                break
        im = im.resize((s, s), Image.BILINEAR)
        if random.random() < 0.5:
            im = im.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.3:                                  # rotasi
            im = im.rotate(random.uniform(-15, 15), Image.BILINEAR, fillcolor=(124, 116, 104))
        if random.random() < 0.7:                                  # brightness/contrast/sat
            for E in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnha

**Peningkatan generalisasi.** Mixup dan CutMix bergantian, label smoothing 0,1,
dropout 0,3, weight decay 0,05, AdamW dengan OneCycleLR, mixed precision.


In [6]:
print("MIXUP / CUTMIX - dipilih acak 50:50 tiap batch")
print(inspect.getsource(T.mixup_cutmix))


MIXUP / CUTMIX - dipilih acak 50:50 tiap batch
def mixup_cutmix(x, yj, yk, alpha=0.4, p_cut=0.5):
    """Satu batch dicampur; label ikut tercampur untuk KEDUA sumbu."""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    if np.random.rand() < p_cut:                                   # CutMix
        H, W = x.shape[2:]
        rh, rw = int(H * (1 - lam) ** 0.5), int(W * (1 - lam) ** 0.5)
        cy, cx = np.random.randint(H), np.random.randint(W)
        y0, y1 = max(cy - rh // 2, 0), min(cy + rh // 2, H)
        x0, x1 = max(cx - rw // 2, 0), min(cx + rw // 2, W)
        x[:, :, y0:y1, x0:x1] = x[idx][:, :, y0:y1, x0:x1]
        lam = 1 - ((y1 - y0) * (x1 - x0) / (H * W))
    else:                                                          # Mixup
        x = lam * x + (1 - lam) * x[idx]
    return x, (yj, yj[idx]), (yk, yk[idx]), lam



## 5. Spesialis per jenis bencana

Komponen yang paling banyak menaikkan skor. Diagnosis OOF menunjukkan bahwa setelah
`_jenis` mencapai 0,994, **91,5% sisa kesalahan** adalah tertukar antar tingkat yang
bersebelahan — dan komposisinya berbeda jauh per jenis:

| | RINGAN↔SEDANG | SEDANG↔BERAT |
|---|---|---|
| BANJIR | 15,5% | 10,1% |
| GEMPA | 3,9% | 14,3% |
| KEBAKARAN | 0,6% | 9,3% |

Satu backbone bersama harus melayani ketiga pola ini sekaligus. Karena itu tiap jenis
diberi ConvNeXt-Tiny sendiri, dilatih hanya pada gambar jenis tersebut.


In [7]:
# Spesialis per jenis: satu ConvNeXt-Tiny untuk tiap bencana, fold sama persis
# sehingga bisa dibandingkan baris per baris dengan model global.
g = K[MODEL.index("ConvNeXt-T@224")]
s_ = K[MODEL.index("ConvNeXt-T per jenis")]
print("AKURASI KERUSAKAN, baris yang sama")
print(f"{'jenis':>12} | {'n':>6} | {'global':>8} | {'spesialis':>10} | {'selisih':>8}")
print("-" * 56)
for j in (1, 2, 3):
    m = jen == j
    a, b = (g[m].argmax(1)+1 == y[m]).mean(), (s_[m].argmax(1)+1 == y[m]).mean()
    print(f"{NJ[j]:>12} | {int(m.sum()):>6} | {a:>8.4f} | {b:>10.4f} | {b-a:>+8.4f}")
a, b = (g.argmax(1)+1 == y).mean(), (s_.argmax(1)+1 == y).mean()
print("-" * 56)
print(f"{'SEMUA':>12} | {len(y):>6} | {a:>8.4f} | {b:>10.4f} | {b-a:>+8.4f}")
print()
print("Naik di ketiga jenis, paling besar di BANJIR - jenis dengan batas paling kabur.")
print("Baris yang jenisnya salah ditebak sengaja dinetralkan (nilai 1/3 rata), supaya")
print("skor OOF tidak lebih optimistis daripada kondisi data uji.")


AKURASI KERUSAKAN, baris yang sama
       jenis |      n |   global |  spesialis |  selisih
--------------------------------------------------------
      BANJIR |   5957 |   0.7156 |     0.7623 |  +0.0467
  GEMPA BUMI |   5746 |   0.7468 |     0.7800 |  +0.0332
   KEBAKARAN |   5779 |   0.8837 |     0.9098 |  +0.0261
--------------------------------------------------------
       SEMUA |  17482 |   0.7814 |     0.8169 |  +0.0355

Naik di ketiga jenis, paling besar di BANJIR - jenis dengan batas paling kabur.
Baris yang jenisnya salah ditebak sengaja dinetralkan (nilai 1/3 rata), supaya
skor OOF tidak lebih optimistis daripada kondisi data uji.


## 6. Keenam cabang


In [8]:
print("KEENAM CABANG, validasi silang level kejadian")
print(f"{'model':>28} | {'_jenis':>8} | {'_kerusakan':>11}")
print("-" * 54)
for nm, a, b in zip(MODEL, J, K):
    u = (a.argmax(1) + 1 == jen).mean()
    us = f"{u:>8.4f}" if "per jenis" not in nm else f"{'-':>8}"
    print(f"{nm:>28} | {us} | {(b.argmax(1)+1 == y).mean():>11.4f}")
print("-" * 54)
print("\nFitur BEKU unggul di _jenis, model yang di-FINE-TUNE unggul jauh di")
print("_kerusakan. Karena itu bobot dicari TERPISAH untuk tiap sumbu.")


KEENAM CABANG, validasi silang level kejadian
                       model |   _jenis |  _kerusakan
------------------------------------------------------
               EffNet B0@224 |   0.9854 |      0.7684
               EffNet B0@320 |   0.9881 |      0.7588
              ConvNeXt-T@224 |   0.9847 |      0.7814
               DINOv3 3-arah |   0.9928 |      0.6743
              DINOv3 ordinal |   0.9928 |      0.6757
        ConvNeXt-T per jenis |        - |      0.8169
------------------------------------------------------

Fitur BEKU unggul di _jenis, model yang di-FINE-TUNE unggul jauh di
_kerusakan. Karena itu bobot dicari TERPISAH untuk tiap sumbu.


## 7. Penggabungan

Bobot non-negatif dicari dengan sampling Dirichlet lalu diperbaiki lokal,
**seluruhnya pada validasi silang data latih**. Label data uji tidak dipakai dalam
pemilihan apa pun.


In [9]:
def cari(P, t, iters=4000, seed=0):
    """Bobot non-negatif, sampling Dirichlet lalu diperbaiki setempat.
    Sama persis dengan scripts/cari_bobot.py - urutan pemakaian rng ikut menentukan
    hasilnya, jadi tidak boleh berbeda sedikit pun."""
    rng = np.random.default_rng(seed)
    best = (0.0, None)
    for _ in range(iters):
        w = rng.dirichlet(np.ones(len(P)) * 0.7)
        a = (sum(wi * Pi for wi, Pi in zip(w, P)).argmax(1) + 1 == t).mean()
        if a > best[0]:
            best = (a, w)
    a, w = best
    for _ in range(300):                                   # perbaikan setempat
        w2 = np.clip(w + rng.normal(0, 0.05, len(w)), 0, None)
        w2 /= w2.sum()
        a2 = (sum(wi * Pi for wi, Pi in zip(w2, P)).argmax(1) + 1 == t).mean()
        if a2 > a:
            a, w = a2, w2
    return a, w

aj, wj = cari(J, jen, seed=1)
ak, wk = cari(K, y, seed=2)
print(f"{'model':>28} | {'bobot _jenis':>13} | {'bobot _kerusakan':>17}")
print("-" * 64)
for i, nm in enumerate(MODEL):
    print(f"{nm:>28} | {wj[i]:>13.2f} | {wk[i]:>17.2f}")
print("-" * 64)
print(f"{'akurasi CV':>28} | {aj:>13.4f} | {ak:>17.4f}")
print(f"\nmicro F1 validasi silang: {(aj+ak)/2:.4f}")


                       model |  bobot _jenis |  bobot _kerusakan
----------------------------------------------------------------
               EffNet B0@224 |          0.03 |              0.13
               EffNet B0@320 |          0.00 |              0.12
              ConvNeXt-T@224 |          0.49 |              0.12
               DINOv3 3-arah |          0.40 |              0.00
              DINOv3 ordinal |          0.07 |              0.00
        ConvNeXt-T per jenis |          0.00 |              0.64
----------------------------------------------------------------
                  akurasi CV |        0.9942 |            0.8223

micro F1 validasi silang: 0.9082


Untuk data uji ditambahkan **TTA**: flip horizontal × dua skala (ukuran latih dan
1,15×), dirata-ratakan atas kelima fold tiap model. Gambar diarahkan ke spesialis
sesuai jenis **prediksi**, dengan aturan pengarah yang sama seperti saat validasi.

## 8. Submission


In [10]:
# Bangun ulang submission dari probabilitas data uji, lalu BUKTIKAN identik
# dengan berkas yang benar-benar dikumpulkan (skor leaderboard 0.844).
z = np.load(O / "test_probs_all.npz", allow_pickle=True)
ids = [str(i) for i in z["ids"]]
TJ = [z[f"{i}_J"] for i in range(len(MODEL))]
TK = [z[f"{i}_K"] for i in range(len(MODEL))]

pj = sum(w * P for w, P in zip(wj, TJ)).argmax(1) + 1
pk = sum(w * P for w, P in zip(wk, TK)).argmax(1) + 1
sol, _ = L.read_solution(next(DATA.rglob("*olution*.csv")))
sub = L.build_submission(sol, ids, pj, np.array([ALF[v] for v in pk]))
L.verify_submission(sub, sol)

asli = pd.read_csv(ROOT / "output_final" / "final_perjenis.csv", dtype=str)
sama = (sub.Target.values.astype(str) == asli.Target.values).all()
print(f"identik dengan final_perjenis.csv yang disubmit: {sama}")
assert sama, "hasil bangun ulang tidak cocok dengan berkas yang disubmit"

print("\nsebaran prediksi data uji (angka sesuai penomoran folder):")
print("  _jenis    : " + "  ".join(f"{NJ[v]}={v} {float((pj==v).mean()):.3f}" for v in (1,2,3)))
print("  _kerusakan: " + "  ".join(f"{NK[k]}={ALF[k]} {float((pk==k).mean()):.3f}" for k in (1,2,3)))
print(f"\n{sub.head(6).to_string(index=False)}")


OK  900 baris (450 _jenis + 450 _kerusakan), urutan & nilai valid.
identik dengan final_perjenis.csv yang disubmit: True

sebaran prediksi data uji (angka sesuai penomoran folder):
  _jenis    : BANJIR=1 0.311  GEMPA BUMI=2 0.356  KEBAKARAN=3 0.333
  _kerusakan: RINGAN=2 0.351  SEDANG=3 0.242  BERAT=1 0.407

         ID Target
    1_jenis      1
1_kerusakan      1
    2_jenis      1
2_kerusakan      1
    3_jenis      1
3_kerusakan      1


## 9. Yang dicoba dan ditolak

Semuanya diuji dengan sungguh-sungguh lalu dibuang. Dicatat karena hasil negatif sama
informatifnya — dan karena beberapa di antaranya terlihat menjanjikan di validasi.

| Gagasan | Hasil |
|---|---|
| **Pembersihan label berbasis keyakinan model** | Naik 0,030 di validasi, **turun 0,027** di kenyataan. Melingkar: "meragukan" ditentukan model yang sama yang kemudian dinilai. |
| **Pembuangan sampel berbasis konsensus antar arsitektur** | Naik di probe, tapi tidak menambah apa-apa di atas spesialis per jenis |
| **Boundary expert** (expert biner untuk batas bersebelahan) | 1 dari 5 lolos guard pada anchor lama; pada model final tidak lagi berpengaruh |
| **ConvNeXt-Small** | 0,8760 < ConvNeXt-Tiny 0,8841 — batch kecil karena GPU 4 GB membuat gradien berisik |
| **Perataan prediksi antar gambar uji yang mirip** | −0,03 s/d −0,13. Gambar berkelompok per *pemandangan*, bukan per keparahan. |
| **Pseudo-labeling di atas fitur beku** | −0,05 s/d −0,09 |
| **Menyamakan label duplikat pHash** | Membantu di probe, nol di ensemble |
| **Kepala kerusakan per jenis di atas fitur beku** | Naik di validasi, datar di kenyataan — beda dengan spesialis penuh di bagian 5 |
| **Resolusi 448 px pada fitur beku** | Tanpa efek; 37,5% gambar aslinya lebih kecil dari 448 px |
| **Gradient reversal (DANN), kalibrasi prior, CORAL** | Tanpa efek atau negatif |

Pelajaran yang berulang: **kenaikan di validasi silang tidak cukup**. Setiap gagasan
hanya diterima kalau validasi dan bukti independen bergerak searah.

## 10. Keterbatasan

1. **Label tingkat kerusakan tidak selalu konsisten.** 17.482 gambar hanya menghasilkan
   15.040 pHash unik; dari 6.682 pasangan gambar identik, 725 (10,9%) diberi label
   kerusakan berbeda. Pemeriksaan manual menemukan pemandangan tanpa kerusakan di dalam
   folder kerusakan berat.
2. **Batas antar tingkat memang kabur.** Beda "sedang" dan "berat" sering merupakan
   penilaian yang bahkan manusia bisa berbeda pendapat.
3. **`_kerusakan` jauh tertinggal dari `_jenis`** (0,82 vs 0,99). Membedakan jenis
   bencana cukup dari pemandangan; menilai keparahan menuntut membaca objek dan
   aktivitas di dalamnya.
4. **Keterbatasan perangkat.** GPU 4 GB membatasi ke model kelas Tiny/B0 pada 224–320 px.

### Catatan kepatuhan

- Hanya dataset resmi panitia (17.482 gambar latih / 450 uji), terlacak satu per satu.
- Tanpa data eksternal dan tanpa API klasifikasi eksternal; seluruh komputasi lokal.
- Bobot pretrained: `efficientnet_b0`, `convnext_tiny` (Apache-2.0),
  `vit_large_patch16_dinov3` — sesuai ketentuan yang mengizinkan pretrained model.
- Seed dikunci (42) untuk pembagian fold dan seluruh proses acak.
